# Rollout degradation by model

This notebook reads a training JSONL file, selects the final epoch for each model/variant, and plots test rollout (R^2) at each relative prediction step. Step 0 is shown on the axis as the observed starting point; no prediction is scored at step 0.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager

font_manager.fontManager = font_manager._load_fontmanager(try_read_cache=False)
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
PAPER_FONT = next((name for name in ("Source Sans 3", "Source Sans Pro", "Nimbus Sans", "Ubuntu Sans", "Liberation Sans", "DejaVu Sans") if name in available_fonts), "DejaVu Sans")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [PAPER_FONT],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 600,
})
print(f"Publication font: {PAPER_FONT}")


In [ ]:
JSONL_PATH = Path("..") / "derived" / "results" / "first_order_diffusion_seed0_rolltrain5_rollhorizon20.jsonl"
TARGET_BLOCK = "rollout_test"
METRIC = "edge_r2"
MAX_STEP = 20
EXPORT_STEM = Path("..") / "derived" / "figures" / "diffusion_rollout_by_model"

with JSONL_PATH.open(encoding="utf-8") as handle:
    rows = [json.loads(line) for line in handle if line.strip()]

if not rows:
    raise ValueError(f"No JSONL records found in {JSONL_PATH}")

# Keep the latest epoch that actually contains per-step rollout metrics.
# This is robust to JSONL files that contain appended runs with different horizons.
final_rows = {}
for row in rows:
    run = row.get("run", "unknown")
    step_block = row.get(TARGET_BLOCK, {}).get("rollout_by_step", {})
    if not step_block:
        continue
    if run not in final_rows or row.get("epoch", -1) >= final_rows[run].get("epoch", -1):
        final_rows[run] = row

curves = {}
persistence_curves = []
missing = []
for run, row in final_rows.items():
    step_block = row.get(TARGET_BLOCK, {}).get("rollout_by_step", {})
    if not step_block:
        missing.append(run)
        continue
    curves[run] = {int(step): values.get(METRIC) for step, values in step_block.items() if int(step) <= MAX_STEP}
    persistent_block = row.get(TARGET_BLOCK, {}).get("rollout_persistent_by_step", {})
    if persistent_block:
        persistence_curves.append({int(step): values.get(METRIC) for step, values in persistent_block.items() if int(step) <= MAX_STEP})
    print(f"{run}: selected epoch {row.get('epoch')} with {len(curves[run])} rollout steps")

if missing:
    print("No per-step rollout metrics for:", ", ".join(missing))
if not curves:
    raise ValueError(
        "This file contains no per-step rollout metrics. Its evaluation split is likely shorter than the requested horizon. "
        "Regenerate with more --num-bins (for example 192) or a smaller --rollout-horizon."
    )

# Persistence is data-defined, so it should agree across models. Average available copies defensively.
persistence_curve = {}
for step in range(1, MAX_STEP + 1):
    values = [curve[step] for curve in persistence_curves if step in curve and curve[step] is not None]
    if values:
        persistence_curve[step] = sum(values) / len(values)

print(f"Loaded {len(curves)} model curves from final epoch records.")
print(f"Persistence baseline: {len(persistence_curve)} rollout steps.")


In [ ]:
colors = plt.get_cmap("tab20").colors
fig, axis = plt.subplots(figsize=(6.5, 3.0), constrained_layout=True)
if persistence_curve:
    persistence_steps = sorted(persistence_curve)
    axis.plot(
        persistence_steps,
        [persistence_curve[step] for step in persistence_steps],
        label="persistence",
        color="0.2",
        linewidth=1.8,
        linestyle="--",
        zorder=3,
    )

for index, (run, values) in enumerate(sorted(curves.items())):
    steps = sorted(values)
    axis.plot(steps, [values[step] for step in steps], label=run, linewidth=1.35, color=colors[index % len(colors)])

axis.set_xlim(0, MAX_STEP)
axis.set_xticks(range(0, MAX_STEP + 1, 5))
axis.set_xlabel("rollout step")
axis.set_ylabel("test rollout $R^2$")
axis.set_title("Diffusion rollout performance by model", loc="left")
axis.axhline(0.0, color="0.45", linewidth=0.7, linestyle=":")
axis.grid(axis="y", color="0.88", linewidth=0.55)
axis.grid(axis="x", visible=False)
axis.spines[["top", "right"]].set_visible(False)
axis.legend(frameon=False, ncol=2, loc="best", handlelength=1.7)

EXPORT_STEM.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(EXPORT_STEM.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(EXPORT_STEM.with_suffix(".png"), dpi=600, bbox_inches="tight")
print(f"Saved {EXPORT_STEM.with_suffix('.pdf')} and {EXPORT_STEM.with_suffix('.png')}")
plt.show()
